## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [1]:
import pandas as pd
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# базова та очікувана конверсія
p1 = 0.19
p2 = 0.20

# ефект
effect_size = proportion_effectsize(p1, p2)

analysis = NormalIndPower()

sample_size_per_group = analysis.solve_power(
    effect_size=effect_size,
    power=0.80,
    alpha=0.05,
    ratio=1
)

total_sample_size = sample_size_per_group * 2

print(f"На групу: {sample_size_per_group:.0f}")
print(f"Всього: {total_sample_size:.0f}")

На групу: 24638
Всього: 49276


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [2]:
df = pd.read_csv('cookie_cats.csv')

In [3]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [5]:
retention7 = (
    df.groupby('version')['retention_7']
    .mean()
    .reset_index()
)

retention7['retention_7'] = retention7['retention_7'] * 100
retention7

,version,retention_7
0,gate_30,19.020134
1,gate_40,18.200004


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [6]:
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

# Розділяємо групи
control = df[df['version'] == 'gate_30']
treatment = df[df['version'] == 'gate_40']

# Кількість успіхів (retention_7 = True)
successes = [
    control['retention_7'].sum(),
    treatment['retention_7'].sum()
]

# Розмір вибірок
nobs = [
    len(control),
    len(treatment)
]

# z-тест для двох пропорцій
z_stat, p_value = proportions_ztest(successes, nobs)

# 95% довірчі інтервали
control_ci = proportion_confint(
    successes[0],
    nobs[0],
    alpha=0.05,
    method='normal'
)

treatment_ci = proportion_confint(
    successes[1],
    nobs[1],
    alpha=0.05,
    method='normal'
)

print(f"z statistic: {z_stat:.4f}")
print(f"p-value: {p_value:.4f}")
print(f"Довірчий інтервал 95% для групи control: [{control_ci[0]:.4f}, {control_ci[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{treatment_ci[0]:.4f}, {treatment_ci[1]:.4f}]")

z statistic: 3.1644
p-value: 0.0016
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]


Ми відхиляємо нульову гіпотезу про рівність показників утримання. Різниця між версіями гри є статистично значущою.

Довірчі інтервали практично не перетинаються. Це свідчить про те, що різниця між показниками утримання є реальною, а не випадковою флуктуацією вибірки.

Отже, версія gate_30 демонструє вищий показник retention_7, ніж gate_40, перенесення воріт на 40-й рівень призвело до погіршення довгострокового утримання користувачів.

4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


Н0: Версія гри та утримання на 7-й день є незалежними.

Н1: Версія гри та утримання на 7-й день є залежними.

In [7]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(
    df['version'],
    df['retention_7']
)

print(contingency_table)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"Chi-square statistic: {chi2:.4f}")
print(f"p-value: {p_value:.6f}")
print(f"Degrees of freedom: {dof}")

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279
Chi-square statistic: 9.9591
p-value: 0.001601
Degrees of freedom: 1


Ми відхиляємо нульову гіпотезу про незалежність змінних (p-value: 0.001601).